In [1]:
import os
import sys
from dataclasses import dataclass
from typing import Tuple, Optional, Literal

import torch
from torch import nn
import torch.nn.functional as F

from model import Linear, MoE

In [2]:
@dataclass
class ModelArgs:
    dim: int = 64
    inter_dim: int = 342
    moe_inter_dim: int = 44
    n_routed_experts: int = 16
    n_shared_experts: int = 2
    n_activated_experts: int = 3
    n_expert_groups: int = 1
    n_limited_groups: int = 1
    score_func: Literal["softmax", "sigmoid"] = "sigmoid"
    route_scale: float = 1.0

args= ModelArgs()
args

ModelArgs(dim=64, inter_dim=342, moe_inter_dim=44, n_routed_experts=16, n_shared_experts=2, n_activated_experts=3, n_expert_groups=1, n_limited_groups=1, score_func='sigmoid', route_scale=1.0)

In [3]:
dtype = torch.bfloat16
torch.set_default_dtype(dtype)
torch.set_num_threads(8)
torch.manual_seed(965)

In [4]:
device = "cuda:0"
with torch.device(device):
    moe_layer = MoE(args)

In [5]:
def init_weights_normal(m):
    if isinstance(m, Linear):
        nn.init.normal_(m.weight, mean=0.0, std=0.1)

moe_layer.apply(init_weights_normal);

In [6]:
batch_size = 8
x = torch.randn(batch_size, args.dim, device=device, dtype=dtype)
with torch.inference_mode():
    y = moe_layer(x)
print(y.shape)

torch.Size([8, 64])


In [7]:
y.max(), y.min()

(tensor(1.3281, device='cuda:0'), tensor(-1.4141, device='cuda:0'))

In [8]:
# TODO: implement MoE.static_forward to get the same output